In [1]:
from pathlib import Path

data_path = Path("/opt/examples/datasets/orders_cdc")

files = sorted(data_path.glob("*.csv"))

for file in files:
    print(file.name)

assert len(files) == 3, "Üç CSV dosyası bulunmalı!"

batch_2026-01-01.csv
batch_2026-01-02.csv
batch_2026-01-03.csv


In [2]:
for file in files:
    print(f"\n{'=' * 50}")
    print(f"FILE: {file.name}")
    print("=" * 50)

    with file.open("r", encoding="utf-8-sig") as f:
        print(f.read())


FILE: batch_2026-01-01.csv
order_id,customer_id,status,amount,updated_at,op
1001,1,NEW,100.00,2026-01-01 09:00:00,I
1002,2,NEW,250.50,2026-01-01 09:05:00,I
1003,3,NEW,75.00,2026-01-01 09:10:00,I
1004,4,NEW,300.00,2026-01-01 09:20:00,I
1005,5,NEW,40.00,2026-01-01 09:30:00,I


FILE: batch_2026-01-02.csv
order_id,customer_id,status,amount,updated_at,op
1001,1,PAID,100.00,2026-01-02 10:00:00,U
1001,1,SHIPPED,100.00,2026-01-02 15:00:00,U
1002,2,PAID,250.50,2026-01-02 10:05:00,U
1002,2,PAID,250.50,2026-01-02 10:05:00,U
1003,3,CANCELLED,75.00,2026-01-02 11:00:00,D
1006,6,NEW,60.00,2026-01-02 11:30:00,I


FILE: batch_2026-01-03.csv
order_id,customer_id,status,amount,updated_at,op
1001,1,PAID,100.00,2026-01-02 12:00:00,U
1002,2,SHIPPED,250.50,2026-01-03 09:00:00,U
1003,3,NEW,75.00,2026-01-01 09:10:00,I
1004,4,PAID,320.00,2026-01-03 09:30:00,U
1007,7,NEW,90.00,2026-01-03 10:00:00,I



In [5]:
import os
from getpass import getpass
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    DecimalType, TimestampType
)
from pyspark.sql.window import Window

# Credentials (docker_compose'daki değerleri kullan)
polaris_credential = (
    os.getenv("POLARIS_CREDENTIAL")
    or getpass("Polaris credential (user:password): ")
)

rustfs_access_key = os.getenv(
    "RUSTFS_ACCESS_KEY", "dataops"
)

# Credentials (docker_compose'daki değerleri kullan)
rustfs_secret_key = (
    os.getenv("RUSTFS_SECRET_ACCESS_KEY")
    or getpass("RustFS secret access key: ")
)

spark = (
    SparkSession.builder
    .appName("Week8-Iceberg-Idempotent-Merge")
    .master("local[2]")

    # Iceberg dependencies
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.12.0,"
        "org.apache.iceberg:iceberg-aws-bundle:1.12.0"
    )

    # Iceberg SQL extensions
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions."
        "IcebergSparkSessionExtensions"
    )

    # Polaris REST Catalog
    .config(
        "spark.sql.catalog.lakehouse",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config(
        "spark.sql.catalog.lakehouse.uri",
        "http://polaris:8181/api/catalog"
    )
    .config(
        "spark.sql.catalog.lakehouse.warehouse",
        "lakehouse"
    )
    .config(
        "spark.sql.catalog.lakehouse.credential",
        polaris_credential
    )
    .config(
        "spark.sql.catalog.lakehouse.scope",
        "PRINCIPAL_ROLE:ALL"
    )

    # RustFS S3 configuration
    .config(
        "spark.sql.catalog.lakehouse.s3.endpoint",
        "http://rustfs:9000"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.path-style-access",
        "true"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.access-key-id",
        rustfs_access_key
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.secret-access-key",
        rustfs_secret_key
    )
    .config(
        "spark.sql.catalog.lakehouse.client.region",
        "us-east-1"
    )
    .getOrCreate()
)

print("Spark Version:", spark.version)

Polaris credential (user:password):  ········
RustFS secret access key:  ········


Spark Version: 4.1.3


26/11/05 12:55:20 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
#testing catalog connection
spark.sql(
    "SHOW NAMESPACES IN lakehouse"
).show(truncate=False)

26/11/05 12:55:37 WARN AuthManagers: Inferring rest.auth.type=oauth2 since property credential was provided. Please explicitly set rest.auth.type to avoid this warning.
26/11/05 12:55:37 WARN OAuth2Manager: Iceberg REST client is missing the OAuth2 server URI configuration and defaults to http://polaris:8181/api/catalog/v1/oauth/tokens. This automatic fallback will be removed in a future Iceberg release. It is recommended to configure the OAuth2 endpoint using the 'oauth2-server-uri' property to be prepared. This warning will disappear if the OAuth2 endpoint is explicitly configured. See https://github.com/apache/iceberg/issues/10537


+---------+
|namespace|
+---------+
+---------+



In [7]:
spark.sql("""
    CREATE NAMESPACE IF NOT EXISTS lakehouse.homework
""")

spark.sql("""
    SHOW NAMESPACES IN lakehouse
""").show(truncate=False)

+---------+
|namespace|
+---------+
|homework |
+---------+



In [8]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.homework.orders (
        order_id INT,
        customer_id INT,
        status STRING,
        amount DECIMAL(10,2),
        updated_at TIMESTAMP,
        is_deleted BOOLEAN
    )
    USING iceberg
""")

DataFrame[]

In [9]:
spark.sql("""
    DESCRIBE TABLE lakehouse.homework.orders
""").show(truncate=False)

spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM lakehouse.homework.orders
""").show()

+-----------+-------------+-------+
|col_name   |data_type    |comment|
+-----------+-------------+-------+
|order_id   |int          |NULL   |
|customer_id|int          |NULL   |
|status     |string       |NULL   |
|amount     |decimal(10,2)|NULL   |
|updated_at |timestamp    |NULL   |
|is_deleted |boolean      |NULL   |
+-----------+-------------+-------+



[Stage 0:===========================================================(1 + 0) / 1]

+----------+
|total_rows|
+----------+
|         0|
+----------+



In [18]:
cdc_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("amount", DecimalType(10, 2), True),
    StructField("updated_at", TimestampType(), True),
    StructField("op", StringType(), True)
])

In [19]:
import re

DATA_PATH = "/opt/examples/datasets/orders_cdc"

def ingest_batch(batch_date, table):

    # 1. Batch dosyasının yolunu oluştur
    assert re.fullmatch(r"\d{4}-\d{2}-\d{2}", batch_date)
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", table)

    file_path = f"{DATA_PATH}/batch_{batch_date}.csv"
    target_table = f"lakehouse.homework.{table}"

    # 2. CSV dosyasını explicit schema ile oku
    raw_df = (
        spark.read
        .option("header", True)
        .schema(cdc_schema)
        .csv(file_path)
    )

    # 3. Her order_id için en güncel değişikliği seç
    window_spec = (
        Window
        .partitionBy("order_id")
        .orderBy(F.col("updated_at").desc())
    )

    dedup_df = (
        raw_df
        .withColumn(
            "rn",
            F.row_number().over(window_spec)
        )
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    # 4. SQL MERGE için Temporary View oluştur
    dedup_df.createOrReplaceTempView("cdc_updates")

    # 5. Iceberg MERGE
    spark.sql(f"""
        MERGE INTO {target_table} AS target
        USING cdc_updates AS source

        ON target.order_id = source.order_id

        WHEN MATCHED
             AND source.updated_at > target.updated_at
        THEN UPDATE SET
            target.customer_id = source.customer_id,
            target.status = source.status,
            target.amount = source.amount,
            target.updated_at = source.updated_at,
            target.is_deleted =
                CASE
                    WHEN source.op = 'D' THEN true
                    ELSE false
                END

        WHEN NOT MATCHED
        THEN INSERT (
            order_id,
            customer_id,
            status,
            amount,
            updated_at,
            is_deleted
        )
        VALUES (
            source.order_id,
            source.customer_id,
            source.status,
            source.amount,
            source.updated_at,
            CASE
                WHEN source.op = 'D' THEN true
                ELSE false
            END
        )
    """)

    print(f"Batch {batch_date} ingested into {target_table}")

In [20]:
spark.table("lakehouse.homework.orders").count()

0

In [21]:
ingest_batch("2026-01-01", "orders")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".                
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

Batch 2026-01-01 ingested into lakehouse.homework.orders


In [22]:
orders_df = spark.table("lakehouse.homework.orders")

orders_df.orderBy("order_id").show(truncate=False)

[Stage 13:>                                                         (0 + 1) / 1]

+--------+-----------+------+------+-------------------+----------+
|order_id|customer_id|status|amount|updated_at         |is_deleted|
+--------+-----------+------+------+-------------------+----------+
|1001    |1          |NEW   |100.00|2026-01-01 09:00:00|false     |
|1002    |2          |NEW   |250.50|2026-01-01 09:05:00|false     |
|1003    |3          |NEW   |75.00 |2026-01-01 09:10:00|false     |
|1004    |4          |NEW   |300.00|2026-01-01 09:20:00|false     |
|1005    |5          |NEW   |40.00 |2026-01-01 09:30:00|false     |
+--------+-----------+------+------+-------------------+----------+



In [23]:
assert orders_df.count() == 5
assert orders_df.filter(F.col("is_deleted") == False).count() == 5

print("Task 2 - First batch validation passed!")

Task 2 - First batch validation passed!


In [24]:
snapshots = spark.sql("""
    SELECT
        snapshot_id,
        committed_at
    FROM lakehouse.homework.orders.snapshots
    ORDER BY committed_at DESC
""")

snapshots.show(truncate=False)

first_snapshot_id = snapshots.first()["snapshot_id"]

print("First Snapshot ID:", first_snapshot_id)

+-------------------+-----------------------+
|snapshot_id        |committed_at           |
+-------------------+-----------------------+
|4008015541683348346|2026-11-05 13:26:07.187|
+-------------------+-----------------------+

First Snapshot ID: 4008015541683348346


In [25]:

ingest_batch("2026-01-02", "orders")
ingest_batch("2026-01-03", "orders")

Batch 2026-01-02 ingested into lakehouse.homework.orders
Batch 2026-01-03 ingested into lakehouse.homework.orders


In [29]:
orders_df = spark.table("lakehouse.homework.orders")

orders_df.orderBy("order_id").show(truncate=False)

+--------+-----------+---------+------+-------------------+----------+
|order_id|customer_id|status   |amount|updated_at         |is_deleted|
+--------+-----------+---------+------+-------------------+----------+
|1001    |1          |SHIPPED  |100.00|2026-01-02 15:00:00|false     |
|1002    |2          |SHIPPED  |250.50|2026-01-03 09:00:00|false     |
|1003    |3          |CANCELLED|75.00 |2026-01-02 11:00:00|true      |
|1004    |4          |PAID     |320.00|2026-01-03 09:30:00|false     |
|1005    |5          |NEW      |40.00 |2026-01-01 09:30:00|false     |
|1006    |6          |NEW      |60.00 |2026-01-02 11:30:00|false     |
|1007    |7          |NEW      |90.00 |2026-01-03 10:00:00|false     |
+--------+-----------+---------+------+-------------------+----------+



In [30]:
# Toplam kayıt sayısı
assert orders_df.count() == 7

# Aktif kayıt sayısı
assert orders_df.filter(F.col("is_deleted") == False).count() == 6

# Silinmiş kayıt sayısı
assert orders_df.filter(F.col("is_deleted") == True).count() == 1

print("Task 3: All row count assertions passed!")

Task 3: All row count assertions passed!


In [31]:
orders = "lakehouse.homework.orders"

# Before re-ingestion
before = (
    spark.table(orders)
    .orderBy("order_id")
    .collect()
)

# Re-ingest the same batch
ingest_batch("2026-01-02", "orders")

# After re-ingestion
after = (
    spark.table(orders)
    .orderBy("order_id")
    .collect()
)

assert before == after, "Idempotency test failed!"

print("Task 4a: Idempotency test passed!")

Batch 2026-01-02 ingested into lakehouse.homework.orders
Task 4a: Idempotency test passed!


In [32]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.homework.orders_replay (
        order_id INT,
        customer_id INT,
        status STRING,
        amount DECIMAL(10,2),
        updated_at TIMESTAMP,
        is_deleted BOOLEAN
    )
    USING iceberg
""")

spark.sql("""
    DELETE FROM lakehouse.homework.orders_replay
""")

assert spark.table(
    "lakehouse.homework.orders_replay"
).count() == 0

In [33]:
ingest_batch("2026-01-03", "orders_replay")
ingest_batch("2026-01-01", "orders_replay")
ingest_batch("2026-01-02", "orders_replay")
ingest_batch("2026-01-03", "orders_replay")

Batch 2026-01-03 ingested into lakehouse.homework.orders_replay
Batch 2026-01-01 ingested into lakehouse.homework.orders_replay
Batch 2026-01-02 ingested into lakehouse.homework.orders_replay
Batch 2026-01-03 ingested into lakehouse.homework.orders_replay


In [34]:
orders_df = spark.table("lakehouse.homework.orders")

replay_df = spark.table(
    "lakehouse.homework.orders_replay"
)

replay_df.orderBy("order_id").show(truncate=False)

assert replay_df.count() == 7

assert orders_df.exceptAll(replay_df).count() == 0
assert replay_df.exceptAll(orders_df).count() == 0

print("Task 4b: Replay test passed!")

+--------+-----------+---------+------+-------------------+----------+
|order_id|customer_id|status   |amount|updated_at         |is_deleted|
+--------+-----------+---------+------+-------------------+----------+
|1001    |1          |SHIPPED  |100.00|2026-01-02 15:00:00|false     |
|1002    |2          |SHIPPED  |250.50|2026-01-03 09:00:00|false     |
|1003    |3          |CANCELLED|75.00 |2026-01-02 11:00:00|true      |
|1004    |4          |PAID     |320.00|2026-01-03 09:30:00|false     |
|1005    |5          |NEW      |40.00 |2026-01-01 09:30:00|false     |
|1006    |6          |NEW      |60.00 |2026-01-02 11:30:00|false     |
|1007    |7          |NEW      |90.00 |2026-01-03 10:00:00|false     |
+--------+-----------+---------+------+-------------------+----------+

Task 4b: Replay test passed!


In [35]:
spark.sql("""
    SELECT
        snapshot_id,
        committed_at,
        operation
    FROM lakehouse.homework.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4008015541683348346|2026-11-05 13:26:07.187|append   |
|6099396512803406608|2026-11-05 13:33:10.774|overwrite|
|1173492170078414175|2026-11-05 13:33:16.382|overwrite|
|8455531948338442285|2026-11-05 13:38:51.156|overwrite|
+-------------------+-----------------------+---------+



In [36]:
# Task 5 - Iceberg Time Travel

old_df = (
    spark.read
    .format("iceberg")
    .option("versionAsOf", str(first_snapshot_id))
    .load("lakehouse.homework.orders")
)

old_df.orderBy("order_id").show(truncate=False)

+--------+-----------+------+------+-------------------+----------+
|order_id|customer_id|status|amount|updated_at         |is_deleted|
+--------+-----------+------+------+-------------------+----------+
|1001    |1          |NEW   |100.00|2026-01-01 09:00:00|false     |
|1002    |2          |NEW   |250.50|2026-01-01 09:05:00|false     |
|1003    |3          |NEW   |75.00 |2026-01-01 09:10:00|false     |
|1004    |4          |NEW   |300.00|2026-01-01 09:20:00|false     |
|1005    |5          |NEW   |40.00 |2026-01-01 09:30:00|false     |
+--------+-----------+------+------+-------------------+----------+



In [37]:
assert old_df.count() == 5

actual = {
    r.order_id: (
        r.customer_id,
        r.status,
        str(r.amount),
        r.updated_at.strftime("%Y-%m-%d %H:%M:%S"),
        r.is_deleted
    )
    for r in old_df.collect()
}

expected = {
    1001: (1, "NEW", "100.00", "2026-01-01 09:00:00", False),
    1002: (2, "NEW", "250.50", "2026-01-01 09:05:00", False),
    1003: (3, "NEW", "75.00", "2026-01-01 09:10:00", False),
    1004: (4, "NEW", "300.00", "2026-01-01 09:20:00", False),
    1005: (5, "NEW", "40.00", "2026-01-01 09:30:00", False)
}

assert actual == expected, "Time Travel sonuçları eşleşmiyor!"

print("Task 5: Time Travel test passed!")

Task 5: Time Travel test passed!


In [38]:
def create_test_table(table):
    assert table in ("orders_naive", "orders_nodedup")

    target = f"lakehouse.homework.{table}"

    spark.sql(f"DROP TABLE IF EXISTS {target}")

    spark.sql(f"""
        CREATE TABLE {target} (
            order_id INT,
            customer_id INT,
            status STRING,
            amount DECIMAL(10,2),
            updated_at TIMESTAMP,
            is_deleted BOOLEAN
        )
        USING iceberg
    """)

    assert spark.table(target).count() == 0
    print(f"Created empty table: {target}")


create_test_table("orders_naive")
create_test_table("orders_nodedup")

Created empty table: lakehouse.homework.orders_naive
Created empty table: lakehouse.homework.orders_nodedup


In [39]:
def ingest_test_batch(
    batch_date,
    table,
    deduplicate=True,
    check_timestamp=True
):
    assert table in ("orders_naive", "orders_nodedup")

    file_path = (
        f"/opt/examples/datasets/orders_cdc/"
        f"batch_{batch_date}.csv"
    )

    target = f"lakehouse.homework.{table}"

    # Read with explicit schema
    source_df = (
        spark.read
        .option("header", True)
        .schema(cdc_schema)
        .csv(file_path)
    )

    # Optional deduplication
    if deduplicate:
        w = (
            Window.partitionBy("order_id")
            .orderBy(F.col("updated_at").desc())
        )

        source_df = (
            source_df
            .withColumn("rn", F.row_number().over(w))
            .filter(F.col("rn") == 1)
            .drop("rn")
        )

    source_df.createOrReplaceTempView("test_cdc_updates")

    # Optional timestamp protection
    timestamp_condition = (
        "AND source.updated_at > target.updated_at"
        if check_timestamp else ""
    )

    spark.sql(f"""
        MERGE INTO {target} AS target
        USING test_cdc_updates AS source
        ON target.order_id = source.order_id

        WHEN MATCHED {timestamp_condition}
        THEN UPDATE SET
            customer_id = source.customer_id,
            status = source.status,
            amount = source.amount,
            updated_at = source.updated_at,
            is_deleted = (source.op = 'D')

        WHEN NOT MATCHED
        THEN INSERT (
            order_id, customer_id, status,
            amount, updated_at, is_deleted
        )
        VALUES (
            source.order_id,
            source.customer_id,
            source.status,
            source.amount,
            source.updated_at,
            (source.op = 'D')
        )
    """)

    print(f"Processed: {batch_date} → {table}")

In [40]:
for batch_date in [
    "2026-01-01",
    "2026-01-02",
    "2026-01-03"
]:
    ingest_test_batch(
        batch_date,
        "orders_naive",
        deduplicate=True,
        check_timestamp=False
    )

    naive_df = spark.table(
    "lakehouse.homework.orders_naive"
)

naive_df.orderBy("order_id").show(truncate=False)

Processed: 2026-01-01 → orders_naive
Processed: 2026-01-02 → orders_naive
Processed: 2026-01-03 → orders_naive
+--------+-----------+-------+------+-------------------+----------+
|order_id|customer_id|status |amount|updated_at         |is_deleted|
+--------+-----------+-------+------+-------------------+----------+
|1001    |1          |PAID   |100.00|2026-01-02 12:00:00|false     |
|1002    |2          |SHIPPED|250.50|2026-01-03 09:00:00|false     |
|1003    |3          |NEW    |75.00 |2026-01-01 09:10:00|false     |
|1004    |4          |PAID   |320.00|2026-01-03 09:30:00|false     |
|1005    |5          |NEW    |40.00 |2026-01-01 09:30:00|false     |
|1006    |6          |NEW    |60.00 |2026-01-02 11:30:00|false     |
|1007    |7          |NEW    |90.00 |2026-01-03 10:00:00|false     |
+--------+-----------+-------+------+-------------------+----------+



In [41]:
#1001 SHIPPED durumundan eski PAID durumuna döndü; 1003 ise silinmiş olmasına rağmen tekrar aktif oldu.

In [42]:
assert naive_df.count() == 7

rows = {
    r.order_id: r
    for r in naive_df.collect()
}

assert rows[1001].status == "PAID"
assert str(rows[1001].amount) == "100.00"
assert rows[1001].updated_at.strftime(
    "%Y-%m-%d %H:%M:%S"
) == "2026-01-02 12:00:00"
assert rows[1001].is_deleted == False

assert rows[1003].status == "NEW"
assert str(rows[1003].amount) == "75.00"
assert rows[1003].updated_at.strftime(
    "%Y-%m-%d %H:%M:%S"
) == "2026-01-01 09:10:00"
assert rows[1003].is_deleted == False

print("Task 6a: Naive MERGE test passed!")

Task 6a: Naive MERGE test passed!


In [43]:
ingest_test_batch(
    "2026-01-02",
    "orders_nodedup",
    deduplicate=False,
    check_timestamp=True
)

nodedup_df = spark.table(
    "lakehouse.homework.orders_nodedup"
)

assert nodedup_df.count() == 6

print("Task 6b: First load contains 6 rows.")

Processed: 2026-01-02 → orders_nodedup
Task 6b: First load contains 6 rows.


In [44]:
#ilk yükleme başarılı olmalı çünkü target tablo başlangıçta boştu. Dolayısıyla gelen 6 satırın tamamı INSERT edilir.

In [45]:
merge_failed = False

try:
    ingest_test_batch(
        "2026-01-02",
        "orders_nodedup",
        deduplicate=False,
        check_timestamp=True
    )

except Exception as e:
    error_message = str(e)
    print("Caught expected MERGE error:")
    print(error_message[:1500])

    merge_failed = (
        "MERGE_CARDINALITY_VIOLATION" in error_message
    )

assert merge_failed, (
    "Expected MERGE_CARDINALITY_VIOLATION was not raised!"
)

print("Task 6b: Expected MERGE failure verified!")

26/11/05 13:56:21 ERROR Utils: Aborting task
org.apache.spark.SparkRuntimeException: [MERGE_CARDINALITY_VIOLATION] The ON search condition of the MERGE statement matched a single row from the target table with multiple rows of the source table.
This could result in the target row being operated on more than once with an update or delete operation and is not allowed. SQLSTATE: 23K01
	at org.apache.spark.sql.errors.QueryExecutionErrors$.mergeCardinalityViolationError(QueryExecutionErrors.scala:2896)
	at org.apache.spark.sql.execution.datasources.v2.MergeRowsExec$BitmapCardinalityValidator.validate(MergeRowsExec.scala:188)
	at org.apache.spark.sql.execution.datasources.v2.MergeRowsExec$MergeRowIterator.next(MergeRowsExec.scala:227)
	at org.apache.spark.sql.execution.datasources.v2.MergeRowsExec$MergeRowIterator.next(MergeRowsExec.scala:200)
	at scala.collection.Iterator$$anon$6.hasNext(Iterator.scala:488)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask$IteratorWithMetri

Caught expected MERGE error:
[MERGE_CARDINALITY_VIOLATION] The ON search condition of the MERGE statement matched a single row from the target table with multiple rows of the source table.
This could result in the target row being operated on more than once with an update or delete operation and is not allowed. SQLSTATE: 23K01
Task 6b: Expected MERGE failure verified!


26/11/05 13:56:21 ERROR DataAndMetadataWritingSparkTask: Aborted commit for partition 0 (task 129, attempt 0, stage 191.0)
26/11/05 13:56:21 ERROR Executor: Exception in task 0.0 in stage 191.0 (TID 129)
org.apache.spark.SparkRuntimeException: [MERGE_CARDINALITY_VIOLATION] The ON search condition of the MERGE statement matched a single row from the target table with multiple rows of the source table.
This could result in the target row being operated on more than once with an update or delete operation and is not allowed. SQLSTATE: 23K01
	at org.apache.spark.sql.errors.QueryExecutionErrors$.mergeCardinalityViolationError(QueryExecutionErrors.scala:2896)
	at org.apache.spark.sql.execution.datasources.v2.MergeRowsExec$BitmapCardinalityValidator.validate(MergeRowsExec.scala:188)
	at org.apache.spark.sql.execution.datasources.v2.MergeRowsExec$MergeRowIterator.next(MergeRowsExec.scala:227)
	at org.apache.spark.sql.execution.datasources.v2.MergeRowsExec$MergeRowIterator.next(MergeRowsExec.sc

In [46]:
assert spark.table(
    "lakehouse.homework.orders_nodedup"
).count() == 6

print("Task 6b: Table unchanged after failed MERGE!")

Task 6b: Table unchanged after failed MERGE!
